In [0]:
"""
Telco Churn Prediction Notebook

This notebook implements an end-to-end machine learning workflow for predicting customer churn in a telco dataset using Databricks, MLflow, and Feature Store. The workflow includes data loading, feature engineering, model training, evaluation, and model registration.

Steps:
1. Parameterize notebook inputs using Databricks widgets (environment, data paths, experiment/model names).
2. Load raw training data from a Delta table and select relevant columns.
3. Create FeatureLookups to enrich raw data with features from the Feature Store.
4. Build a training dataset by merging raw data and features.
5. Display the training dataset for inspection.
6. Set up MLflow experiment tracking and registry URIs.
7. Define a scikit-learn pipeline with preprocessing, feature selection, and classifier.
8. Perform hyperparameter tuning using GridSearchCV.
9. Split data into training and validation sets.
10. Train the model and evaluate performance.
11. Log the trained model to MLflow, including feature lookup metadata and model signature.
12. Register the model in Unity Catalog and set model alias.
13. Save training and validation datasets as Delta tables for downstream use.

Parameters:
- env: Notebook environment (dev, staging, prod)
- training_data_raw: Path to raw training data Delta table
- experiment_name: MLflow experiment name
- model_name: Registered model name in Unity Catalog
- features_table: Feature Store table name
- catalog_name: Catalog name for saving output tables

Outputs:
- Registered ML model with feature metadata
- Training and validation datasets saved as Delta tables
- Model URI and version for deployment

Dependencies:
- Databricks Feature Engineering Client
- MLflow
- scikit-learn

"""

In [0]:
# List of input args needed to run this notebook as a job.
# Provide them via DB widgets or notebook arguments.

# Notebook Environment
dbutils.widgets.dropdown("env", "dev", ["dev","staging", "prod"], "Environment Name")
env = dbutils.widgets.get("env")

# Path to the Hive-registered Delta table containing the training data.
dbutils.widgets.text(
    "training_data_raw",
    "telco_churn_train_raw",
    label="Path to the training data",
)

# MLflow experiment name.
dbutils.widgets.text(
    "experiment_name",
    f"/Workspace/Shared/mlops_talk/telco_churn_model",
    label="MLflow experiment name",
)


# Unity Catalog registered model name to use for the trained mode.
dbutils.widgets.text(
    "model_name", "telco_churn_model", label="Full (Three-Level) Model Name"
)

# Pickup features table name
dbutils.widgets.text(
    "features_table",
    "telco_cust_features",
    label="Features Table",
)

dbutils.widgets.text(
    "catalog_name",
    "mlops_dbx_talk_dev",
    label="Catalog Name",
)

dbutils.widgets.text(
    "username",
    "",
    label="Username",
)

In [0]:
username = dbutils.widgets.get("username")
if username=="" or username is None:
    raise Exception("Provide the username")
catalog_name = dbutils.widgets.get("catalog_name")

input_table_name = f"{catalog_name}.{username}.{dbutils.widgets.get('training_data_raw')}"
experiment_name = f"/Workspace/Shared/mlops_talk_{username}/{dbutils.widgets.get('experiment_name')}"
model_name = f"{catalog_name}.{username}.{dbutils.widgets.get('model_name')}"

In [0]:
import pyspark.sql.functions as f

In [0]:
import time
import mlflow
import mlflow.spark
import mlflow.pyspark.ml
import pyspark.sql.functions as f

from databricks.feature_engineering import FeatureLookup, FeatureEngineeringClient
from mlflow.client import MlflowClient

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    StandardScaler,
    StringIndexer,
    UnivariateFeatureSelector,
    VectorAssembler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [0]:
mlflow.set_tracking_uri('databricks')
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name)
mlflow.set_experiment(experiment_name)

In [0]:
raw_data = spark.table(input_table_name).select(
        f.col("customerID").alias("customer_id"),
        f.when(f.col("Churn") == "Yes", f.lit(1)).otherwise(f.lit(0)).cast("int").alias("churn")
    )
raw_data.display()

In [0]:
features_table = f"{catalog_name}.{username}.{dbutils.widgets.get("features_table")}"
 
 
feature_lookups = [
    FeatureLookup(
        table_name=features_table,
        feature_names=None,
        lookup_key=["customer_id"],
    ),
]

In [0]:


# Since the rounded timestamp columns would likely cause the model to overfit the data
# unless additional feature engineering was performed, exclude them to avoid training on them.

fe = FeatureEngineeringClient()

# Create the training set that includes the raw input data merged with corresponding features from both feature tables
training_set = fe.create_training_set(
    df=raw_data, # specify the df 
    feature_lookups=feature_lookups, 
    label="churn",
)


# Load the TrainingSet into a dataframe which can be passed into sklearn for training a model
training_df = training_set.load_df()

In [0]:
# Display the training dataframe, and note that it contains both the raw input data and the features from the Feature Store
training_df.display()

In [0]:
numeric_features = [
    "tenure_months",
    "tenure_years",
    "monthly_charges",
    "total_charges_filled",
    "avg_monthly_charge_lifetime",
    "abs_charges_gap",
]

categorical_features = [
    "gender",
    "internet_service",
    "contract_type",
    "payment_method",
    "tenure_bucket",
    "monthly_charge_bucket",
]

required_columns = ["customer_id", "churn"] + numeric_features + categorical_features
missing_columns = [c for c in required_columns if c not in training_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns in training_df: {missing_columns}")

train_df, test_df = training_df.limit(100).randomSplit([0.8, 0.2], seed=123)

# train_df.write.mode("overwrite").saveAsTable(f"{catalog_name}.churn.telco_churn_train")
# test_df.write.mode("overwrite").saveAsTable(f"{catalog_name}.churn.telco_churn_validation")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

In [0]:
# -----------------------------
# Preprocessing pipeline
# -----------------------------
#
# Original sklearn notebook:
#   numeric -> StandardScaler
#   categorical -> OrdinalEncoder
#   feature_selection -> SelectKBest
#   classifier -> LogisticRegression
#
# Spark rewrite:
#   numeric -> Imputer -> VectorAssembler
#   categorical -> StringIndexer -> OneHotEncoder
#   all -> VectorAssembler -> StandardScaler
#   classifier -> LogisticRegression
#
# NOTE:
# We intentionally skip feature selection in this first rewrite.
# If you want, we can later add an optional UnivariateFeatureSelector stage.

imputed_numeric_cols = [f"{c}_imputed" for c in numeric_features]
indexed_categorical_cols = [f"{c}_idx" for c in categorical_features]
encoded_categorical_cols = [f"{c}_ohe" for c in categorical_features]

# imputer = Imputer(
#     inputCols=numeric_features,
#     outputCols=imputed_numeric_cols,
#     strategy="median",
# )

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep",
    )
    for c in categorical_features
]

encoder = OneHotEncoder(
    inputCols=indexed_categorical_cols,
    outputCols=encoded_categorical_cols,
    handleInvalid="keep",
)

assembler = VectorAssembler(
    inputCols=numeric_features + encoded_categorical_cols,
    outputCol="assembled_features",
    handleInvalid="keep",
)

scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="features",
    withMean=False,
    withStd=True,
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="churn",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=500,
    regParam=0.0,
    elasticNetParam=0.0,
    standardization=False,  # already scaling upstream
)

pipeline_stages =  indexers + [encoder, assembler, scaler, lr]

pipeline = Pipeline(stages=pipeline_stages)

# Optional selector example (disabled on purpose):
#
# selector = UnivariateFeatureSelector(
#     featuresCol="features",
#     outputCol="selected_features",
#     labelCol="churn",
#     featureType="continuous",
#     labelType="categorical",
#     selectionMode="numTopFeatures",
#     selectionThreshold=30,
# )
#
# lr = LogisticRegression(
#     featuresCol="selected_features",
#     labelCol="churn",
#     predictionCol="prediction",
#     probabilityCol="probability",
#     rawPredictionCol="rawPrediction",
#     maxIter=500,
#     regParam=0.0,
#     elasticNetParam=0.0,
#     standardization=False,
# )
#
# pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, scaler, selector, lr])

In [0]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="churn",
    predictionCol="prediction",
    metricName="accuracy",
)

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.maxIter, [3])
    .build()
)

crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=accuracy_evaluator,
    numFolds=2,
    seed=123,
    parallelism=1,
)

mlflow.end_run()

# mlflow.spark.autolog(
#     log_models=False,
#     log_input_examples=False,
#     log_model_signatures=False,
# )
SPARKML_TMP = "/Volumes/mlops_dbx_talk_dev/churn/helpers/pysparkml/"
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = SPARKML_TMP

with mlflow.start_run() as run:
    cv_model = crossval.fit(train_df)
    best_model = cv_model.bestModel

In [0]:
# Recover best parameter map from the CV results
best_index = max(range(len(cv_model.avgMetrics)), key=lambda i: cv_model.avgMetrics[i])
best_param_map = cv_model.getEstimatorParamMaps()[best_index]

best_params = {param.name: value for param, value in best_param_map.items()}
best_cv_accuracy = cv_model.avgMetrics[best_index]

print("Best params:", best_params)
print("Best CV accuracy:", best_cv_accuracy)

test_predictions = best_model.transform(test_df)

accuracy = accuracy_evaluator.evaluate(test_predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="churn",
    predictionCol="prediction",
    metricName="f1",
).evaluate(test_predictions)

auc_roc = BinaryClassificationEvaluator(
    labelCol="churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(test_predictions)

auc_pr = BinaryClassificationEvaluator(
    labelCol="churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
).evaluate(test_predictions)

mlflow.log_metric("cv_best_accuracy", best_cv_accuracy)
mlflow.log_metric("test_accuracy", accuracy)
mlflow.log_metric("test_f1", f1)
mlflow.log_metric("test_auc_roc", auc_roc)
mlflow.log_metric("test_auc_pr", auc_pr)

for k, v in best_params.items():
    mlflow.log_param(f"best_{k}", v)

display(
    test_predictions.select(
        "customer_id",
        "churn",
        "prediction",
        "probability",
        "rawPrediction",
    )
)

In [0]:
# Package the best Spark pipeline model with feature lookup metadata.
# This is what allows batch / serving inference to automatically retrieve features.
SPARKML_TMP = "/Volumes/mlops_dbx_talk_dev/churn/helpers/pysparkml_models/"
os.environ["MLFLOW_DFS_TMP"] = SPARKML_TMP

model_info = fe.log_model(
    model=best_model,
    artifact_path="model_packaged",
    flavor=mlflow.spark,
    training_set=training_set,
    registered_model_name=model_name,
)

mlflow.log_param("model_flavor", "mlflow.spark")
mlflow.log_param("feature_table", features_table)

mlflow.end_run()

In [0]:
client = MlflowClient()
run_id = model_info.run_id

def find_version_by_run(model_name, run_id, max_wait_s=60):
    for _ in range(max_wait_s):
        for mv in client.search_model_versions(f"name='{model_name}'"):
            if getattr(mv, "run_id", None) == run_id:
                return int(mv.version)
        time.sleep(1)
    return None

version = getattr(model_info, "registered_model_version", None)
if version is None:
    version = find_version_by_run(model_name, run_id)

if version is None:
    raise RuntimeError(
        f"Could not resolve model version for model={model_name} and run_id={run_id}"
    )

client.set_registered_model_alias(
    name=model_name,
    alias="staging",
    version=version,
)

print(f"Registered model version: {version}")
print(f"Alias 'staging' -> version {version}")